1. Imports

In [1]:
import os
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

print("PyTorch and torchvision imported successfully.")

PyTorch and torchvision imported successfully.


2. Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

Device: cpu
Using CPU


3. Dataset + preprocessing 

In [3]:
train_dir = r"D:\deep learning project\data\intel image classification\seg_train\seg_train"
test_dir = r"D:\deep learning project\data\intel image classification\seg_test\seg_test"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(
    train_dir,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=transform
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Training: 11227
Validation: 2807
Test: 3000


4. DataLoaders

In [4]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 351
Validation batches: 88
Test batches: 94


5. oad pretrained ViT

In [5]:
model = models.vit_b_16(
    weights=models.ViT_B_16_Weights.DEFAULT
)

# Freeze the pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Replace the classification head
model.heads.head = nn.Linear(
    model.heads.head.in_features,
    6
)

model = model.to(device)

print(model)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\hp/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth


100%|██████████| 330M/330M [01:39<00:00, 3.49MB/s] 


VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine

6. Parameter count

In [6]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 85803270
Trainable parameters: 4614


7. Loss and optimizer

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.heads.head.parameters(),
    lr=0.001
)

print("Loss and optimizer configured.")

Loss and optimizer configured.


8. Training setup

In [8]:
EPOCHS = 5

train_accuracies = []
val_accuracies = []

best_val_accuracy = 0.0

best_model_path = r"D:\deep learning project\results\models\vit_best.pth"

print("Training setup ready.")

Training setup ready.


9. training 

In [9]:
for epoch in range(EPOCHS):

    start_time = time.time()

    # =========================
    # Training
    # =========================
    model.train()

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total

    # =========================
    # Validation
    # =========================
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct / total

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Training Accuracy: {train_accuracy:.2f}% "
        f"Validation Accuracy: {val_accuracy:.2f}% "
        f"Time: {elapsed/60:.1f} min"
    )

    # =========================
    # Save best model
    # =========================
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("Best ViT model saved.")

Epoch [1/5] Training Accuracy: 88.26% Validation Accuracy: 91.34% Time: 59.3 min
Best ViT model saved.
Epoch [2/5] Training Accuracy: 92.17% Validation Accuracy: 92.20% Time: 54.5 min
Best ViT model saved.
Epoch [3/5] Training Accuracy: 93.16% Validation Accuracy: 92.55% Time: 51.8 min
Best ViT model saved.
Epoch [4/5] Training Accuracy: 93.49% Validation Accuracy: 92.34% Time: 54.7 min
Epoch [5/5] Training Accuracy: 93.93% Validation Accuracy: 92.20% Time: 45.7 min


10. Load best model

In [10]:
# Load the best saved ViT model

best_model_path = r"D:\deep learning project\results\models\vit_best.pth"

model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model = model.to(device)
model.eval()

print("Best ViT model loaded successfully.")

Best ViT model loaded successfully.


10. Final test evaluation

In [11]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Test accuracy
test_accuracy = 100 * accuracy_score(all_labels, all_predictions)

print(f"ViT Test Accuracy: {test_accuracy:.2f}%")
print(f"Test samples: {len(all_labels)}")

print("\nViT Classification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=test_dataset.classes,
        digits=4
    )
)

ViT Test Accuracy: 92.17%
Test samples: 3000

ViT Classification Report:
              precision    recall  f1-score   support

   buildings     0.9289    0.9268    0.9278       437
      forest     0.9895    0.9979    0.9937       474
     glacier     0.9072    0.7957    0.8478       553
    mountain     0.8264    0.9067    0.8647       525
         sea     0.9635    0.9824    0.9728       510
      street     0.9307    0.9381    0.9344       501

    accuracy                         0.9217      3000
   macro avg     0.9244    0.9246    0.9235      3000
weighted avg     0.9227    0.9217    0.9212      3000

